# Practical 4: Generative Adversarial Network (GAN) using Keras

**Aim:** Build a small GAN that learns to generate Fashion-MNIST-like images.

This notebook is intentionally beginner-friendly: short code blocks, plain explanations, and only the main steps needed for the practical.

## What is a GAN?

A GAN contains two neural networks:

- **Generator:** starts with random numbers and tries to create a believable image.
- **Discriminator:** looks at an image and predicts whether it is real (from the dataset) or generated (fake).

They train against each other. Over time, the Generator tries to fool the Discriminator.

**Dataset:** Fashion-MNIST, containing 28 × 28 grayscale images of clothing items.

**Reference reading**
- [How to Code GAN Training Hacks](https://machinelearningmastery.com/how-to-code-generative-adversarial-network-hacks/)
- [18 Impressive Applications of GANs](https://machinelearningmastery.com/impressive-applications-of-generative-adversarial-networks/)

The first article discusses useful training practices such as scaling images to [-1, 1], using Adam, and label smoothing. The second describes applications such as image generation, image translation, super-resolution, and image inpainting.

## 1. Import libraries

We use TensorFlow's Keras API to define and train the networks, NumPy for random numbers, and Matplotlib to display images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

## 2. Load and prepare the dataset

Fashion-MNIST images originally have pixel values from 0 to 255. We scale them to **[-1, 1]** because the Generator will use `tanh` as its final activation.

We only need the training images: this is an image-generation task, not a clothing-classification task.

In [ ]:
(x_train, _), (_, _) = keras.datasets.fashion_mnist.load_data()

# Add a channel dimension: (number of images, height, width, channels)
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32")

# Scale pixel values from [0, 255] to [-1, 1]
x_train = (x_train - 127.5) / 127.5

print("Training images shape:", x_train.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())

### View some real images

This gives us a visual idea of the images the Generator will learn from.

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle("Real Fashion-MNIST images")
plt.tight_layout()
plt.show()

## 3. Build the Generator

The Generator accepts a vector of 100 random values (called **noise**) and converts it into a 28 × 28 image.

- `Dense`: learns combinations of the random values.
- `LeakyReLU`: activation that allows a small negative output.
- `tanh`: produces pixels in the range [-1, 1], matching our training images.

This is a small fully connected GAN, chosen to keep the practical easy to follow.

In [ ]:
LATENT_DIM = 100  # number of random values used to create one image

generator = keras.Sequential([
    layers.Input(shape=(LATENT_DIM,)),
    layers.Dense(256),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dense(28 * 28, activation="tanh"),
    layers.Reshape((28, 28, 1))
], name="Generator")

generator.summary()

## 4. Build the Discriminator

The Discriminator receives an image and outputs one number between 0 and 1:

- near **1**: it thinks the image is real;
- near **0**: it thinks the image is generated.

`sigmoid` is used for this probability-like output. Binary cross-entropy measures how far its predictions are from the target labels.

In [ ]:
discriminator = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dropout(0.3),
    layers.Dense(256),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dense(1, activation="sigmoid")
], name="Discriminator")

discriminator.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

discriminator.summary()

## 5. Combine both networks

The combined model connects:

**random noise → Generator → Discriminator**

When training this combined model, the Discriminator's weights are frozen. That means the error signal teaches the Generator to create images that the Discriminator considers real.

We compile the Discriminator first, then freeze it for the combined model.

In [ ]:
discriminator.trainable = False

noise_input = keras.Input(shape=(LATENT_DIM,))
generated_image = generator(noise_input)
validity = discriminator(generated_image)

gan = keras.Model(noise_input, validity, name="GAN")
gan.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss="binary_crossentropy"
)

print("GAN is ready to train.")

## 6. Train the GAN

For every batch, we do two updates:

1. **Train the Discriminator:** show it real images labelled `1` and generated images labelled `0`.
2. **Train the Generator:** ask the combined model to make generated images receive target label `1`. Only the Generator is updated in this step.

We use **one-sided label smoothing** for real examples: the target is `0.9` instead of `1.0`. This is a simple technique mentioned in the GAN training-hacks article. GAN training can be unstable, so results may vary between runs.

Start with 10 epochs to test the notebook. You can increase this later if you have time.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128
SAMPLE_COUNT = 10

# Keep a record of average losses for plotting later
generator_losses = []
discriminator_losses = []

# Use batches of real images
dataset = tf.data.Dataset.from_tensor_slices(x_train)
dataset = dataset.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True)

for epoch in range(EPOCHS):
    d_losses = []
    g_losses = []

    for real_images in dataset:
        batch_size = tf.shape(real_images)[0]

        # ----- Step 1: train the Discriminator -----
        noise = tf.random.normal((batch_size, LATENT_DIM))
        fake_images = generator(noise, training=True)

        real_labels = tf.ones((batch_size, 1)) * 0.9  # label smoothing
        fake_labels = tf.zeros((batch_size, 1))

        d_real_loss = discriminator.train_on_batch(real_images, real_labels)
        d_fake_loss = discriminator.train_on_batch(fake_images, fake_labels)
        d_loss = (d_real_loss[0] + d_fake_loss[0]) / 2
        d_losses.append(d_loss)

        # ----- Step 2: train the Generator -----
        noise = tf.random.normal((batch_size, LATENT_DIM))
        misleading_labels = tf.ones((batch_size, 1))
        g_loss = gan.train_on_batch(noise, misleading_labels)
        g_losses.append(float(g_loss))

    generator_losses.append(float(np.mean(g_losses)))
    discriminator_losses.append(float(np.mean(d_losses)))

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Discriminator loss: {discriminator_losses[-1]:.4f} | "
        f"Generator loss: {generator_losses[-1]:.4f}"
    )

## 7. Plot training losses

Losses help us monitor training, but GAN losses do **not** need to decrease smoothly. The Generator and Discriminator are competing, so look at the generated images as well.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(generator_losses, label="Generator loss")
plt.plot(discriminator_losses, label="Discriminator loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN training losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Generate new images

We provide fresh random noise to the trained Generator. The output images are synthetic: they are learned from patterns in the training dataset, not copied labels or hand-drawn images.

The images may look blurry after only a few epochs. If you have enough time or GPU memory, try raising `EPOCHS` to 20–30.

In [ ]:
def show_generated_images(count=16):
    noise = tf.random.normal((count, LATENT_DIM))
    images = generator.predict(noise, verbose=0)

    # Convert pixels from [-1, 1] back to [0, 1] for display
    images = (images + 1) / 2

    side = int(np.ceil(np.sqrt(count)))
    plt.figure(figsize=(side * 2, side * 2))
    for i in range(count):
        plt.subplot(side, side, i + 1)
        plt.imshow(images[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
        plt.axis("off")
    plt.suptitle("Images generated by the GAN")
    plt.tight_layout()
    plt.show()

show_generated_images(16)

## 9. Compare real and generated images

This visual comparison is a simple qualitative evaluation. A small beginner GAN may not generate sharp or recognisable clothing images; that is normal and does not automatically mean the code failed.

In [ ]:
noise = tf.random.normal((10, LATENT_DIM))
fake_images = (generator.predict(noise, verbose=0) + 1) / 2
real_images = (x_train[:10] + 1) / 2

plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(real_images[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    plt.subplot(2, 10, i + 11)
    plt.imshow(fake_images[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    plt.axis("off")

plt.figtext(0.25, 0.92, "Real images", ha="center", fontsize=12)
plt.figtext(0.75, 0.92, "Generated images", ha="center", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

## 10. GAN applications

The applications article describes several uses of GANs. Examples include:

- **Image generation:** create new images resembling examples in a dataset.
- **Image-to-image translation:** transform one kind of image into another, such as sketches into photos.
- **Super-resolution:** create a higher-resolution version of an image.
- **Image inpainting:** fill missing or damaged parts of an image.
- **Photo editing and style changes:** modify selected visual properties.

These applications need suitable datasets and often more advanced models than the small GAN built in this practical.

## 11. Conclusion

In this practical, we:

1. Loaded and normalised the Fashion-MNIST dataset.
2. Built a Generator to create images from random noise.
3. Built a Discriminator to classify images as real or generated.
4. Combined the two models and trained them adversarially using Keras.
5. Plotted the losses and visually inspected generated images.

A GAN learns through competition between the Generator and Discriminator. Training quality depends on the architecture, training time, data, and hyperparameters; realistic results are not guaranteed from a short run.

### References

1. Jason Brownlee, [How to Code GAN Training Hacks](https://machinelearningmastery.com/how-to-code-generative-adversarial-network-hacks/), Machine Learning Mastery.
2. Jason Brownlee, [18 Impressive Applications of Generative Adversarial Networks (GANs)](https://machinelearningmastery.com/impressive-applications-of-generative-adversarial-networks/), Machine Learning Mastery.